In [1]:
from lightning.pytorch.callbacks import ModelCheckpoint
from loaders.Sndataloader2 import SNpart_Dataset
from model.GNN_inf_seg import Lightning_GNN
import torch_geometric as tg
import numpy as np
import lightning as pl
import datetime
import yaml
import os
import wandb
import torch
import pyvista as pv

In [2]:
with open('configs/config_SNpart.yml', 'r') as f:
    config = yaml.safe_load(f)

config['batch_size'] = 10

# Data setup
dataset_test = SNpart_Dataset(root=config['root'],
                                 split='test')

test_loader = tg.loader.DataLoader(dataset_test,
                                  batch_size=config['batch_size'],
                                  num_workers=2,
                                  shuffle=True)

# Model setup
GNN_model = Lightning_GNN(config=config)
GNN_model.load_state_dict(torch.load('/home/lars/models/2024-08-22_20.41.44SN_part_blocks_base_wskip/epoch=199-train_loss=0.12.ckpt')['state_dict'])
GNN_model.to('cpu')
a = 1

In [3]:
with open('configs/config_SNpart_global.yml', 'r') as f:
    config = yaml.safe_load(f)

config['batch_size'] = 10

# Data setup
dataset_test = SNpart_Dataset(root=config['root'],
                                 split='test')

test_loader = tg.loader.DataLoader(dataset_test,
                                  batch_size=config['batch_size'],
                                  num_workers=2,
                                  shuffle=True)

# Model setup
GNN_model_global = Lightning_GNN(config=config)
GNN_model_global.load_state_dict(torch.load('/home/lars/models/2024-09-13_20.13.11SN_part_global_n=15/epoch=180-train_loss=0.11.ckpt')['state_dict'])
GNN_model_global.to('cpu')
a = 1

In [4]:
with torch.no_grad():
    GNN_model.eval()
    sample = next(iter(test_loader))
    out_pc = GNN_model(sample)
    out_pc_g = GNN_model_global(sample)

prediction = torch.argmax(out_pc, dim=1)
prediction_g = torch.argmax(out_pc_g, dim=1)

In [5]:
# extract sample with lowest accuracy   
accr_list = [] 
accr_g_list = []

for i in range(config['batch_size']):
    accr = torch.sum(prediction[sample.batch == i] == sample.y[sample.batch == i]).item() / len(sample.y[sample.batch == i])
    accr_g = torch.sum(prediction_g[sample.batch == i] == sample.y[sample.batch == i]).item() / len(sample.y[sample.batch == i])
    accr_list.append((accr, i))
    accr_g_list.append((accr_g, i))

lowest_accuracy_indices = [x[1] for x in sorted(accr_list, key=lambda x: x[0])[:3]]
print(accr_list)
print(accr_g_list)

[(0.99365234375, 0), (0.9970703125, 1), (0.9052734375, 2), (0.9814453125, 3), (0.9208984375, 4), (0.83984375, 5), (0.9970703125, 6), (0.98974609375, 7), (0.96923828125, 8), (0.9716796875, 9)]
[(0.982421875, 0), (0.98779296875, 1), (0.0, 2), (0.876953125, 3), (0.85546875, 4), (0.80126953125, 5), (0.53076171875, 6), (0.96630859375, 7), (0.779296875, 8), (0.7451171875, 9)]


In [8]:
batch_idx = 8
pos = sample.pos[sample.batch == batch_idx].numpy()
pred = prediction[sample.batch == batch_idx].numpy()
label = sample.y[sample.batch == batch_idx].numpy()
pred_g = prediction_g[sample.batch == batch_idx].numpy()

# Create a PyVista plotter
plotter = pv.Plotter()
point_cloud_pv = pv.PolyData(pos)
point_cloud_pv['labels'] = label
plotter.add_points(point_cloud_pv, scalars='labels', cmap='viridis', render_points_as_spheres=True)
#plotter.camera_position = camera_position
#plotter.focal_point = focal_point
plotter.view_up = [0, 0, 1]
#plotter.view_vector([0.36806499, 0.17783007,  -0.91263609])
plotter.show()

Widget(value='<iframe src="http://localhost:37371/index.html?ui=P_0x736b39ea2060_2&reconnect=auto" class="pyvi…

In [195]:
camera_position = plotter.camera_position
focal_point = plotter.focal_point

In [196]:
# Create a new PyVista plotter for the output point cloud
plotter_output = pv.Plotter()
point_cloud_pv['output_labels'] = pred
plotter_output.add_points(point_cloud_pv, scalars='output_labels', cmap='viridis', render_points_as_spheres=True)
plotter_output.camera_position = camera_position
plotter_output.focal_point = focal_point
plotter_output.view_up = [0, 0, 1]
plotter_output.show()

Widget(value='<iframe src="http://localhost:42151/index.html?ui=P_0x75a5af343c80_100&reconnect=auto" class="py…

In [194]:
# Create a new PyVista plotter for the output point cloud
plotter_output = pv.Plotter()
point_cloud_pv['output_labels'] = pred_g
plotter_output.add_points(point_cloud_pv, scalars='output_labels', cmap='viridis', render_points_as_spheres=True)
plotter_output.camera_position = camera_position
plotter_output.focal_point = focal_point
plotter_output.view_up = [0, 0, 1]
plotter_output.show()

Widget(value='<iframe src="http://localhost:42151/index.html?ui=P_0x75a4b351eb40_99&reconnect=auto" class="pyv…